# Lab 5 — Skew, cache, AQE and UDFs

## 1. Setup

This notebook runs on its own: the cells below recreate the required state from the first part.

In [1]:
import io, re, time, contextlib
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Lab 5 - Performance")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "false")   # off on purpose; we turn it on in section 4
    .config("spark.ui.showConsoleProgress", "false") # hides the progress bars
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark   :", spark.version)
print("Cores   :", spark.sparkContext.defaultParallelism)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/04 18:21:08 WARN Utils: Your hostname, ERJ1752, resolves to a loopback address: 127.0.1.1; using 10.23.20.50 instead (on interface enp0s31f6)
26/11/04 18:21:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 18:21:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark   : 4.0.1
Cores   : 24
Spark UI: http://10.23.20.50:4040


In [2]:
@contextlib.contextmanager
def timer(label):
    spark.sparkContext.setJobDescription(label)
    t0 = time.perf_counter()
    yield # when it finishes
    print(f"{label:<45} {time.perf_counter() - t0:6.2f} s")
    spark.sparkContext.setJobDescription(None)

def run(df):
    df.write.format("noop").mode("overwrite").save()

def plan_text(df, mode="simple"):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode=mode)
    return buf.getvalue()

def count_shuffles(df):
    return len(re.findall(r"\bExchange\b", plan_text(df)))

def partition_balance(df, *keys, n=16, label=""):
    """Repartitions by key and shows the size of the largest partition vs the median."""
    s = (df.repartition(n, *keys)
           .groupBy(F.spark_partition_id()).count()
           .agg(F.max("count").alias("max"), F.percentile_approx("count", 0.5).alias("median"))
           .first())
    print(f"{label:<12} largest = {s['max']:>9,} | median = {s['median']:>9,} | ratio = {s['max'] / s['median']:.1f}x")

In [3]:
N_ORDERS    = 5_000_000
N_CUSTOMERS = 200_000
N_PRODUCTS  = 1_000
REGIONS     = ["North", "Northeast", "Midwest", "Southeast", "South"]
CATEGORIES  = ["Computers", "Peripherals", "Audio", "Storage", "Cables"]
BASE        = "/tmp/lab4"

def pick(values, seed):
    arr = F.array(*[F.lit(v) for v in values])
    return F.element_at(arr, (F.rand(seed) * len(values)).cast("int") + 1)

orders_gen = (
    spark.range(N_ORDERS).withColumnRenamed("id", "order_id")
    .withColumn("customer_id", (F.rand(1) * N_CUSTOMERS).cast("int"))
    .withColumn("product_id",  (F.rand(2) * N_PRODUCTS).cast("int"))
    .withColumn("quantity",    (F.rand(3) * 5 + 1).cast("int"))
    .withColumn("price",       F.round(F.rand(4) * 490 + 10, 2))
    .withColumn("region",      pick(REGIONS, 5))
    .withColumn("order_date",  F.date_add(F.to_date(F.lit("2024-01-01")), (F.rand(6) * 365).cast("int")))
)

customers_gen = (
    spark.range(N_CUSTOMERS).withColumnRenamed("id", "customer_id")
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("segment", pick(["Consumer", "Corporate", "Small Business"], 11))
    .withColumn("signup_year", (F.rand(12) * 6 + 2019).cast("int"))
)

products_gen = (
    spark.range(N_PRODUCTS).withColumnRenamed("id", "product_id")
    .withColumn("product_id", F.col("product_id").cast("int"))
    .withColumn("product_name", F.concat(F.lit("Product "), F.col("product_id")))
    .withColumn("category", pick(CATEGORIES, 21))
)

with timer("Generate and save the 3 tables"):
    orders_gen.write.mode("overwrite").parquet(f"{BASE}/orders")
    customers_gen.write.mode("overwrite").parquet(f"{BASE}/customers")
    products_gen.write.mode("overwrite").parquet(f"{BASE}/products")

orders    = spark.read.parquet(f"{BASE}/orders")
customers = spark.read.parquet(f"{BASE}/customers")
products  = spark.read.parquet(f"{BASE}/products")

orders.show(5)

Generate and save the 3 tables                  2.56 s
+--------+-----------+----------+--------+------+---------+----------+
|order_id|customer_id|product_id|quantity| price|   region|order_date|
+--------+-----------+----------+--------+------+---------+----------+
| 2291666|      73092|       417|       1| 98.89|  Midwest|2024-12-19|
| 2291667|     186251|       989|       3|317.22|    North|2024-01-28|
| 2291668|     140692|       483|       4|183.57|    North|2024-01-20|
| 2291669|      32718|       607|       5| 113.4|    North|2024-07-17|
| 2291670|       1265|       401|       5|411.26|Northeast|2024-04-30|
+--------+-----------+----------+--------+------+---------+----------+
only showing top 5 rows


---
## 2. Data skew (unbalanced data)

Skew happens when a few keys hold a large share of the rows. After the shuffle, **one partition becomes huge** and its task takes much longer than the others. The whole job waits for it.

```
   Tasks after the shuffle (time)
   T1 ███
   T2 ███
   T3 ████████████████████████████████   <- "hot" key
   T4 ███
```

Typical symptom in the Spark UI: in the Stages tab, the maximum task duration is much greater than the median, and the job sits at "199/200 tasks".

Let's simulate it: **40% of the orders** belong to `customer_id = 0` (for example, a marketplace buying on behalf of third parties).

In [4]:
orders_skew = orders.withColumn(
    "customer_id",
    F.when(F.rand(7) < 0.4, F.lit(0)).otherwise(F.col("customer_id"))
)
orders_skew.write.mode("overwrite").parquet(f"{BASE}/orders_skew")
orders_skew = spark.read.parquet(f"{BASE}/orders_skew")

# 1) Detect: which keys hold the most rows?
(
    orders_skew.groupBy("customer_id").count()
    .withColumn("pct", F.round(F.col("count") / N_ORDERS * 100, 2))
    .orderBy(F.desc("count"))
    .show(5)
)

+-----------+-------+----+
|customer_id|  count| pct|
+-----------+-------+----+
|          0|1999763|40.0|
|      32361|     35| 0.0|
|     103645|     35| 0.0|
|     100499|     35| 0.0|
|      72640|     34| 0.0|
+-----------+-------+----+
only showing top 5 rows


In [5]:
# 2) See the effect on partitions after a shuffle by key
(
    orders_skew.repartition(8, "customer_id")
    .groupBy(F.spark_partition_id().alias("partition")).count()
    .orderBy(F.desc("count"))
    .show()
)

+---------+-------+
|partition|  count|
+---------+-------+
|        7|2374060|
|        2| 378545|
|        3| 377244|
|        6| 375019|
|        4| 374651|
|        5| 374332|
|        1| 373141|
|        0| 373008|
+---------+-------+



In [6]:
# 3) A simple metric: largest partition vs median
partition_balance(orders, "customer_id", label="orders")
partition_balance(orders_skew, "customer_id", label="orders_skew")

orders       largest =   317,729 | median =   311,943 | ratio = 1.0x
orders_skew  largest = 2,186,568 | median =   187,215 | ratio = 11.7x


In a cluster with 100 cores, the stage lasts as long as the **largest** task. With a partition 10x bigger than the median, 99 cores wait for one.

> In local mode, with 2 to 8 cores, the total time barely changes: the total work is the same and there are few cores to sit idle. That is why we measure skew here through the partition distribution, which is what the Stages tab shows in production.

### 2.1 Fixes

| Technique | Idea | When |
|--------|-------|--------|
| **Broadcast** | Removes the shuffle, so skew no longer matters | The other side fits in memory |
| **Salting** | Splits the hot key into N artificial subkeys | Both sides are large |
| **AQE skew join** | Spark detects huge partitions and splits them on its own | Spark 3+; see section 4 |
| **Handle the key separately** | Processes the hot key on its own and does a `union` | A few known keys |

**Salting**: we add a random number (`salt`) from 0 to N-1 on the large side, and replicate each row of the small side N times, once per salt. The join becomes `(customer_id, salt)`, and the hot key spreads across N partitions.

```
  orders_skew                     customers (replicated N times)
  cust=0, salt=3   ─────────▶     cust=0, salt=0..N-1
  cust=0, salt=7                  cust=1, salt=0..N-1
  cust=0, salt=1                  ...
```

In [7]:
SALT = 16

orders_salted = orders_skew.withColumn("salt", (F.rand(8) * SALT).cast("int"))
customers_salted = customers.crossJoin(
    spark.range(SALT).select(F.col("id").cast("int").alias("salt"))
)

partition_balance(orders_skew, "customer_id", label="no salt")
partition_balance(orders_salted, "customer_id", "salt", label="with salt")

# The join result is still correct: same number of rows
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
n_plain  = orders_skew.join(customers, "customer_id").count()
n_salted = orders_salted.join(customers_salted, ["customer_id", "salt"]).count()
print(f"\nrows without salt: {n_plain:,} | with salt: {n_salted:,}")
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

no salt      largest = 2,186,568 | median =   187,215 | ratio = 11.7x
with salt    largest =   441,645 | median =   308,298 | ratio = 1.4x

rows without salt: 5,000,000 | with salt: 5,000,000


Salting has a cost: the small side becomes N times larger and there is more data in the shuffle. If the dimension fits in memory, **broadcast is the simplest solution**, because there is no shuffle and the skew stops existing.

---
## 3. Cache and persist

Because of lazy evaluation, **each action recomputes the whole plan from the read onward**. If an expensive DataFrame is used several times, the cache avoids this rework.

| Level (`StorageLevel`) | Where it is stored | Note |
|------------------------|-------------|------------|
| `MEMORY_AND_DISK` | Memory, spilling to disk | Default for `df.cache()` |
| `MEMORY_ONLY` | Memory only | What does not fit is recomputed |
| `DISK_ONLY` | Disk only | Useful when the computation is very expensive |

**Use cache when:** the same DataFrame feeds several actions (reports, loops, iterative ML algorithms).

**Avoid it when:** the DataFrame is used only once, or is just a simple Parquet read (reading again is cheap). The cache takes up memory that shuffles could use.

In [8]:
spark.conf.set("spark.sql.shuffle.partitions", spark.sparkContext.defaultParallelism * 2)

# Expensive to compute but small (one row per customer): join + aggregation with countDistinct
customer_features = (
    orders.join(products, "product_id")
    .groupBy("customer_id")
    .agg(
        F.sum(F.col("price") * F.col("quantity")).alias("revenue"),
        F.count("*").alias("n_orders"),
        F.countDistinct("category").alias("n_categories"),
    )
    .join(customers, "customer_id")
)

# E.g. export one file per segment. Each export is a separate action.
def export_by_segment(df):
    for seg in ["Consumer", "Corporate", "Small Business"]:
        run(df.filter(F.col("segment") == seg))

with timer("3 exports WITHOUT cache"):
    export_by_segment(customer_features)

features_cached = customer_features.cache()
with timer("Materialize the cache (count)"):
    features_cached.count()

with timer("3 exports WITH cache"):
    export_by_segment(features_cached)

_ = features_cached.unpersist()

3 exports WITHOUT cache                         2.44 s
Materialize the cache (count)                   0.78 s
3 exports WITH cache                            0.20 s


`cache()` is lazy too: the DataFrame is stored only on the first action. That is why we materialize it with `count()`. In the Spark UI, the **Storage** tab shows what is cached and how much space it takes.

The cache paid off because the data is **expensive to compute and small to store** (200 thousand rows). In exercise 4 you will see the opposite case.

---
## 4. Adaptive Query Execution (AQE)

Without AQE, the plan is decided **before** execution, using estimates. With AQE (the default since Spark 3.2), Spark **re-optimizes between stages**, using the real statistics of each shuffle.

| AQE feature | What it does | Setting |
|----------------|-----------|--------------|
| Coalesce partitions | Merges small partitions after the shuffle (200 become a few) | `spark.sql.adaptive.coalescePartitions.enabled` |
| Switch join strategy | Converts sort-merge into broadcast if one side became small after filters | `spark.sql.adaptive.autoBroadcastJoinThreshold` |
| Skew join | Splits giant partitions into smaller pieces | `spark.sql.adaptive.skewJoin.enabled` |

So far AQE has been off. Let's turn it on.

In [9]:
spark.conf.set("spark.sql.shuffle.partitions", 200)
agg = orders.groupBy("region").agg(F.sum("price").alias("total"))

spark.conf.set("spark.sql.adaptive.enabled", "false")
print("Without AQE, partitions after the shuffle:", agg.rdd.getNumPartitions())

spark.conf.set("spark.sql.adaptive.enabled", "true")
agg = orders.groupBy("region").agg(F.sum("price").alias("total"))
print("With AQE, partitions after the shuffle:", agg.rdd.getNumPartitions())

Without AQE, partitions after the shuffle: 200
With AQE, partitions after the shuffle: 1


In [10]:
# AQE skew join: we lower the thresholds so that our sample dataset counts as "skewed"
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256KB")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "256KB")

skew_join = (
    orders_skew.join(customers, "customer_id")
    .groupBy("segment").agg(F.count("*").alias("orders"))
)
print("Before running:")
print([l.strip() for l in plan_text(skew_join).splitlines() if "AdaptiveSparkPlan" in l])

skew_join.collect()   # action on the DataFrame itself, so we can see its final plan

print("\nAfter running:")
final_plan = plan_text(skew_join)
print([l.strip() for l in final_plan.splitlines()
       if "AdaptiveSparkPlan" in l or "SortMergeJoin" in l or "AQEShuffleRead" in l][:4])

Before running:
['AdaptiveSparkPlan isFinalPlan=false']

After running:
['AdaptiveSparkPlan isFinalPlan=true', '+- AQEShuffleRead coalesced', '+- *(5) SortMergeJoin [customer_id#56], [customer_id#25], Inner', ':  +- AQEShuffleRead coalesced']


Before execution, the plan shows `isFinalPlan=false`. Afterwards, it shows `isFinalPlan=true` and `SortMergeJoin(skew=true)`: AQE detected the hot partition and split it. `AQEShuffleRead coalesced and skewed` indicates merged and split partitions.

> In production, keep AQE on (the default). It does not replace good design, but it handles much of the partition tuning automatically.

In [11]:
# Restore default settings
for k in ["spark.sql.autoBroadcastJoinThreshold",
          "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
          "spark.sql.adaptive.advisoryPartitionSizeInBytes"]:
    spark.conf.unset(k)

---
## 5. Python UDF vs native functions vs Pandas UDF

Native functions (`F.*`) run inside the JVM and are optimized by Catalyst. A Python UDF forces Spark to **serialize each row, send it to a Python process and bring the result back**.

```
  F.when(...)     JVM ──────────────────────────────▶ JVM            (fast)
  @udf            JVM ─▶ serializes row by row ───▶ Python ─▶ JVM     (slow)
  @pandas_udf     JVM ─▶ Arrow batches (columnar) ▶ Pandas ─▶ JVM     (in between)
```

In [12]:
from pyspark.sql.functions import pandas_udf

def classify(price):
    if price is None: return None
    if price < 100:   return "Budget"
    if price < 300:   return "Mid-range"
    return "Premium"

classify_udf = F.udf(classify, "string")

@pandas_udf("string")
def classify_pandas(price: pd.Series) -> pd.Series:
    return pd.Series(np.select([price < 100, price < 300], ["Budget", "Mid-range"], default="Premium"))

native = (F.when(F.col("price") < 100, "Budget")
           .when(F.col("price") < 300, "Mid-range")
           .otherwise("Premium"))

with timer("Native function (F.when)"):
    run(orders.withColumn("cls", native))
with timer("Pandas UDF (Arrow)"):
    run(orders.withColumn("cls", classify_pandas("price")))
with timer("Python UDF row by row"):
    run(orders.withColumn("cls", classify_udf("price")))

Native function (F.when)                        0.36 s
Pandas UDF (Arrow)                              1.67 s
Python UDF row by row                           0.65 s


Order of preference: **native function > Pandas UDF > Python UDF**. Before writing a UDF, check the [function list](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html): one almost always exists already.

---
## 6. Spark UI: where to look

Open the URL printed in the setup while the cells run. (On Colab the UI is not directly accessible; run locally to explore it.)

| Tab | What to look for |
|-----|----------------|
| **Jobs** | Which actions took longest (the names come from `timer`) |
| **Stages** | Shuffle Read/Write, and **maximum vs median** task duration (skew) |
| **Storage** | Cached DataFrames and how much space they take |
| **SQL / DataFrame** | Visual plan with rows and time per operator |
| **Executors** | Memory, GC time, failed tasks |

---
## 7. Exercises

> Use `count_shuffles`, `explain()` and `timer` to justify each answer.

In [13]:
# ── Exercise 1 ──────────────────────────────────────────────────────────────
# How many shuffles does the query below have? Explain where each one comes from.
# Then rewrite it to remove at least one shuffle without changing the result.

ex1 = (
    orders
    .join(customers, "customer_id")
    .join(products, "product_id")
    .groupBy("segment", "category")
    .agg(F.sum(F.col("price") * F.col("quantity")).alias("revenue"))
    .orderBy(F.desc("revenue"))
)

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)   # simulates "large" dimensions
# Your solution:

In [14]:
# ── Solution 1 ──────────────────────────────────────────────────────────────
print("Original — shuffles:", count_shuffles(ex1))
# 2 sort-merge joins = 4 Exchanges (each side of each join),
# + 1 from groupBy, + 1 from orderBy (range partitioning) = 6.
# (Spark may reuse the partitioning and show slightly fewer.)

ex1_opt = (
    orders
    .join(F.broadcast(customers), "customer_id")
    .join(F.broadcast(products), "product_id")
    .groupBy("segment", "category")
    .agg(F.sum(F.col("price") * F.col("quantity")).alias("revenue"))
    .orderBy(F.desc("revenue"))
)
print("Optimized — shuffles:", count_shuffles(ex1_opt))

with timer("Original"):
    run(ex1)
with timer("With broadcast"):
    run(ex1_opt)

spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

Original — shuffles: 6
Optimized — shuffles: 2
Original                                        1.34 s
With broadcast                                  0.51 s


In [15]:
# ── Exercise 2 ──────────────────────────────────────────────────────────────
# In orders_skew, find:
#   a) what percentage of the rows the most frequent key represents;
#   b) the ratio between the largest and the median partition after repartition(16, "customer_id").
# A ratio above ~5 is a good sign of skew.

# Your solution:
top = orders_skew.groupBy("customer_id").count().orderBy(F.desc("count")).first()
print(f"a) customer_id={top['customer_id']} accounts for {top['count'] / N_ORDERS:.1%} of the rows")

sizes = (
    orders_skew.repartition(16, "customer_id")
    .groupBy(F.spark_partition_id()).count()
    .agg(F.max("count").alias("max"), F.percentile_approx("count", 0.5).alias("median"))
    .first()
)
print(f"b) largest = {sizes['max']:,} | median = {sizes['median']:,} | ratio = {sizes['max'] / sizes['median']:.1f}x")

a) customer_id=0 accounts for 40.0% of the rows
b) largest = 2,186,568 | median = 187,215 | ratio = 11.7x


In [16]:
# ── Exercise 3 ──────────────────────────────────────────────────────────────
# The function below uses a UDF to compute the quarter ("Q1".."Q4") of order_date.
# Rewrite it using only native functions and compare the times.

quarter_udf = F.udf(lambda d: f"Q{(d.month - 1) // 3 + 1}", "string")

# Your solution:

quarter_native = F.concat(F.lit("Q"), F.quarter("order_date"))

with timer("UDF"):
    run(orders.withColumn("q", quarter_udf("order_date")))
with timer("Native (F.quarter)"):
    run(orders.withColumn("q", quarter_native))

orders.select("order_date", quarter_native.alias("q")).show(3)

UDF                                             1.26 s
Native (F.quarter)                              0.56 s
+----------+---+
|order_date|  q|
+----------+---+
|2024-12-19| Q4|
|2024-01-28| Q1|
|2024-01-20| Q1|
+----------+---+
only showing top 3 rows


In [17]:
# ── Exercise 4 ──────────────────────────────────────────────────────────────
# You need to generate 4 reports from:
#   orders filtered on price > 200, joined with customers, with a revenue column.
# Measure the total time with and without cache. In which scenario would the cache NOT be worth it?

# Your solution:
base = (
    orders.filter(F.col("price") > 200)
    .join(customers, "customer_id")
    .withColumn("revenue", F.col("price") * F.col("quantity"))
)

def four_reports(df):
    run(df.groupBy("region").agg(F.sum("revenue")))
    run(df.groupBy("segment").agg(F.sum("revenue")))
    run(df.groupBy("order_date").agg(F.count("*")))
    run(df.groupBy("signup_year").agg(F.avg("price")))

with timer("4 reports without cache"):
    four_reports(base)

with timer("4 reports with cache (incl. materializing)"):
    cached = base.cache()
    cached.count()
    four_reports(cached)
_ = cached.unpersist()

# Here the cache tends NOT to pay off:
# - the data is wide (millions of rows, all columns) and expensive to store;
# - each report uses few columns, and without cache Spark reads only those (column pruning);
# - recomputing (reading Parquet + broadcast join) is cheap.
# Compare with section 3: there the data was small and expensive to compute.

4 reports without cache                         1.45 s
4 reports with cache (incl. materializing)      1.84 s


### Solutions

---
## Summary: symptom, cause and action

| Symptom | Likely cause | What to do |
|---------|----------------|-------------|
| Many `Exchange` operators in the plan | Too many joins and aggregations | Broadcast dimensions, aggregate before joining |
| Job stuck at "199/200 tasks" | Data skew | Broadcast, salting, AQE skew join |
| Thousands of millisecond tasks | Too many partitions | AQE on, tune `shuffle.partitions` |
| Few cores busy | Too few partitions | `repartition`, read smaller files |
| Same computation repeated | Reuse without cache | `cache()` + `unpersist()` |
| Very slow computed column | Python UDF | Native function or Pandas UDF |
| Slow read | CSV/JSON, no pruning | Parquet/Delta, filter and select early |

In [19]:
spark.stop()